# Lab 11: How to gate a RAG change before release

**CPU | API key: `OPENAI_API_KEY` (chat and embedding calls; the release tests need none) | In class**

## Goal

Stop a change before release with tests that read its saved results. The moment that teaches it: a change that lets
support agents use the assistant scores better than today's release, and the tests still fail it, because a note
customers may not read reached the evidence given to customers. Before that, an outdated policy left in the index
after an update keeps answering with the old rule.

Your pipeline now answers follow-ups, and you can measure any change question by question, but every user can read
everything in the index, and nothing stops a change that scores better while breaking a rule. This lab adds
documents customers may not read, and writes tests a change must pass before release.

## What you will learn

- How to label every document with who may read it, and list a user's allowed records by hand.
- How to filter by permission after the search and before it (a metadata filter on the store), and what each does
  to a customer's results.
- How to replace an outdated policy in the index by ID, and check that no old version survives.
- How to write a build manifest and save one row per question for a change.
- How to write release checks as pytest tests (access, versions, refusals, reviewed regressions, quotes), run them on
  saved results and read a failing assertion.
- Why an average cannot pass a hard boundary.

**In class:** Parts A and B. Parts C to F are self-study.

## Set up the notebook

Run these two cells first, every time; Lab 00 explains each line. **Setup** finds the repository folder (on Colab it
copies the repository and installs the packages first). **Config** reads the settings and finds your key.

In [1]:
# Setup: on Colab, copy the repository and install the packages. Everywhere else they are installed already.
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("/content/rag_for_nlp").exists():
        !git clone -q --depth 1 https://github.com/tatwan/rag_for_nlp.git /content/rag_for_nlp
    %cd /content/rag_for_nlp
    %pip install -q -r requirements.txt

ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents] if (folder / "models.env").exists())
sys.path.insert(0, str(ROOT))
print("Repository folder:", ROOT.name)

Repository folder: rag_for_nlp


In [2]:
# Config: settings from models.env (your .env and environment variables win); the key from the environment, .env or Colab Secrets.
from shared.settings import load_settings, load_key, stamp

settings = load_settings()
key_found_in = load_key("OPENAI_API_KEY")

for name, value in stamp(settings).items():
    print(f"{name:18}{value}")
print(f"{'OPENAI_API_KEY':18}{'found in ' + key_found_in if key_found_in else 'not found'}")

chat_model        gpt-4o-mini
reasoning_effort  None
embed_model       text-embedding-3-small
judge_model       gpt-4o-mini
rerank_model      gpt-4o-mini
chat_endpoint     https://api.openai.com/v1
embed_endpoint    https://api.openai.com/v1
run_date          2026-10-24
OPENAI_API_KEY    found in .env


This lab uses `chat_model`, `embed_model` and `JSON_MODE`. It makes 148 short chat calls and embeds the corpus once. The saved
outputs and every count in the text come from `gpt-4o-mini`; the README lists what `gpt-6-luna` did. Your own run will differ a little: model replies vary from run to run and similarity scores can move in the third decimal, so compare the order of results and the kind of reply with the text, not every number.

---

## Part A: Label who may read each document

Part A adds documents that not every user may read, and gives every document in the index a label that says who may.

The support team wants its agents to use the assistant too, with their own documents: three pages of the support
runbook and one note from the incident team. Customers must never read them. They are in
`resources/constructed-records.json`, written for this lab; the five policies in `data/support-case.json` are unchanged.

In [3]:
import json

LAB = ROOT / "11_Release_Gate"
constructed = json.loads((LAB / "resources" / "constructed-records.json").read_text(encoding="utf-8"))
for record in constructed["restricted"]:
    print(record["chunk_id"], f"{record['access']:8}|", record["text"])

USERS = constructed["users"]
print(USERS)

R01 staff   | Support agents can send a reset link that stays valid for 4 hours to an Atlas workspace owner who is locked out. Do not offer it to other users.
R02 staff   | Support agents reset temporary Atlas accounts from the admin console after they confirm the account's end date.
R03 staff   | Support agents can export all reports of an Atlas workspace when its owner asks in writing.
R04 internal| When a customer says an Atlas reset link expired too soon, check the request time in the admin console before you send a new link.
{'customer': ['public'], 'agent': ['public', 'staff', 'internal']}


Each record carries an **access label**, the group of readers it is meant for: `staff` on the runbook pages, `internal`
on the incident note, because each team labels its own documents. `USERS` says which labels each kind of user may
read: a customer `public` only, an agent all three. A list of what a user may read is an **allow-list**: a label that
nobody added to it is read by nobody.

The 272 chunks carry no label at all; only the policies do. Give every record a label when the index is built, so a
filter never has to guess. `{"access": "public", **record}` adds `public` and keeps a label the record already has:

In [4]:
from collections import Counter
from shared import lc_pipeline

records = [{"access": "public", **record} for record in lc_pipeline.load_corpus()] + constructed["restricted"]
print(len(records), "records |", Counter(record["access"] for record in records))

281 records | Counter({'public': 277, 'staff': 3, 'internal': 1})


Before any filter, list by hand what each user may read:

In [5]:
for user, levels in USERS.items():
    allowed = [record["chunk_id"] for record in records if record["access"] in levels]
    print(f"{user:9} may read {len(allowed)} of {len(records)} records")

customer  may read 277 of 281 records
agent     may read 281 of 281 records


Build the store with Lab 05's `build_store()`, which keeps every field but the text as the document's metadata:

In [6]:
store = lc_pipeline.build_store(lc_pipeline.to_documents(records), settings)
print(len(store.store), "documents |", store.get_by_ids(["R04"])[0].metadata)

281 documents | {'chunk_id': 'R04', 'source_id': 'R04', 'title': 'Atlas incident note: expired links', 'version': '2026-10-01', 'access': 'internal'}


`store.store` is the dictionary in which `InMemoryVectorStore` keeps each document by its ID; this lab reads it to list
everything in the index. A store backed by a database is listed with a query instead.

### Checkpoint A

| You should see | Our run |
|---|---|
| The restricted records | R01 to R03 labelled `staff`, R04 labelled `internal` |
| Labels | 277 `public`, 3 `staff`, 1 `internal` |
| Allowed | customer 277 of 281, agent 281 of 281 |
| The store | 281 documents; R04's metadata holds `access: internal` |

---

## Part B: Filter by permission after the search, then before it

Part B keeps restricted documents out of a customer's evidence, in the two places a filter can sit.

A customer asks *Can support send me a new Atlas reset link?*

*Predict first:* with no filter, how many of the three hits will be documents this customer may not read?

Search with no filter:

In [7]:
question = "Can support send me a new Atlas reset link?"
hits = lc_pipeline.retrieve(store, question)
for document, score in hits:
    print(f"{score:.3f} {document.id} {document.metadata['access']:8}| {document.page_content[:70]}")

0.638 R01 staff   | Support agents can send a reset link that stays valid for 4 hours to a
0.634 R04 internal| When a customer says an Atlas reset link expired too soon, check the r
0.561 R02 staff   | Support agents reset temporary Atlas accounts from the admin console a


All three hits are documents the customer may not read. Answer from them three times:

In [8]:
for _ in range(3):
    print(lc_pipeline.answer(question, lc_pipeline.evidence(hits), settings).answer)

Yes, support can send you a new Atlas reset link if you are the workspace owner and are locked out.


Yes, support can send you a new Atlas reset link if you are an Atlas workspace owner who is locked out.


Yes, support can send you a new Atlas reset link if you are the workspace owner and are locked out.


All three replies answer *Yes* and add the runbook's condition, that the customer owns the workspace and is locked out: *Yes, support can send you a new Atlas reset link if you are the workspace owner and are locked out.* The customer has just read a runbook rule. No rule in the prompt can undo that: the text already reached the model,
and every log of the request.

The first place for a filter is after the search: search as before, then drop the hits the user may not read. This
is a **post-filter**. `can_read(user)` returns a filter for one user, a function that is `True` for a document whose
label the user may read:

In [9]:
def can_read(user):
    """A filter for one user: True for a document whose access label the user may read."""
    return lambda document: document.metadata["access"] in USERS[user]

post = [(document, score) for document, score in hits if can_read("customer")(document)]
print("post-filter:", [document.id for document, _ in post])
print(lc_pipeline.answer(question, lc_pipeline.evidence(post), settings).answer)

post-filter: []


Insufficient evidence.


Nothing is left: the three places went to documents the customer may not read, so the customer gets *Insufficient
evidence.* from no evidence at all. How often does that happen? Count the hits a post-filter leaves on Lab 06's 38
golden questions (searches only):

In [10]:
from shared import evals

golden = evals.golden_set()
kept = Counter(sum(can_read("customer")(document) for document, _ in lc_pipeline.retrieve(store, record["question"]))
               for record in golden)
print("hits left of 3, and how many questions:", dict(sorted(kept.items())))

hits left of 3, and how many questions: {1: 5, 2: 2, 3: 31}


The post-filter keeps all three hits for 31 of the 38 questions, two for 2 and only one for 5. Each restricted document that ranks high takes a place a public one could have had.

The second place is before the search: pass the filter to the store, which applies it to every document before it
ranks them (Lab 05, Part G). This is a **pre-filter**:

In [11]:
pre = lc_pipeline.retrieve(store, question, filter=can_read("customer"))
print("pre-filter:", [document.id for document, _ in pre])
for _ in range(3):
    print(lc_pipeline.answer(question, lc_pipeline.evidence(pre), settings).answer)

pre-filter: ['H02', 'H01', 'H04']


Insufficient evidence.


Insufficient evidence.


Insufficient evidence.


The pre-filter fills all three places with documents the customer may read: H02, H01 and H04. All three answers are *Insufficient evidence.* No public
policy says that support sends links (H02 tells the customer to request a new one), so refusing is right here. A
post-filter can always leave fewer than *k* hits; a pre-filter never does while the user may read *k* documents.

A filter is a guard, and a guard is measured on what it should let through as well as on what it should stop. Ask
a question that only the runbook answers, once as an agent and once as a customer:

In [12]:
staff_question = "How long can a support agent make an Atlas reset link last?"
for user in USERS:
    user_hits = lc_pipeline.retrieve(store, staff_question, filter=can_read(user))
    print(user, [document.id for document, _ in user_hits])
    for _ in range(3):
        print("  ", lc_pipeline.answer(staff_question, lc_pipeline.evidence(user_hits), settings).answer)

customer ['H02', 'H01', 'H04']


   A support agent can make an Atlas reset link last for 30 minutes.


   A support agent can make an Atlas reset link last for 30 minutes.


   A support agent can make an Atlas reset link last for 30 minutes.


agent ['R04', 'R01', 'H02']


   A support agent can make an Atlas reset link last for 4 hours.


   Support agents can make an Atlas reset link last for 4 hours.


   Support agents can make an Atlas reset link last for 4 hours.


The agent gets R04, R01 and H02, and *4 hours* in 3 of 3 (*Support agents can make an Atlas reset link last for 4 hours.*, once worded *A support agent can ...*): the filter lets through what it should. The customer gets H02, H01 and H04, and *A support agent can make an Atlas reset link last for 30 minutes.* in 3 of 3. H02's 30 minutes is about every reset link; nothing in it is about agents. The filter did its job, and the model answered from the nearest document the customer may read. Check a
permission filter by reading the replies, not only the hits.

### Checkpoint B

| You should see | Our run |
|---|---|
| No filter | R01, R04, R02; *Yes* with the runbook's condition, 3 of 3 |
| Post-filter | no hits; *Insufficient evidence.*; on the 38 golden questions three hits left for 31, two for 2, one for 5 |
| Pre-filter | H02, H01, H04; *Insufficient evidence.* 3 of 3 |
| The runbook question | agent: R04, R01, H02, *4 hours* 3 of 3; customer: H02, H01, H04, *... last for 30 minutes.* 3 of 3 |

---

## Part C: Replace an outdated policy in the index (self-study)

Part C keeps old versions out of the evidence: a filter by permission cannot see a document that is out of date.

Until 1 October, H02 said that Atlas reset links last 24 hours. Suppose the index was first built in April, from an
export that named each document by its source and its date, and that the October update added today's H02 beside
it. Add the April document to the store:

In [13]:
april = constructed["outdated"]
print(april["chunk_id"], april["version"], "|", april["text"])
store.add_documents(lc_pipeline.to_documents([april]))
print(len(store.store), "documents")

H02-2026-04-01 2026-04-01 | Atlas reset links expire after 24 hours. Request a new link if the old link has expired.


282 documents


Ask Lab 06's two questions about how long a link lasts, as a customer, three times each:

In [14]:
expiry = ["How long does an Atlas reset link remain valid?",
          "When does the password recovery email stop working for Atlas?"]
for q in expiry:
    q_hits = lc_pipeline.retrieve(store, q, filter=can_read("customer"))
    print(q, [document.id for document, _ in q_hits])
    for _ in range(3):
        print("  ", lc_pipeline.answer(q, lc_pipeline.evidence(q_hits), settings).answer)

How long does an Atlas reset link remain valid? ['H02', 'H02-2026-04-01', 'H04']


   Insufficient evidence.


   Insufficient evidence.


   Insufficient evidence.


When does the password recovery email stop working for Atlas? ['H01', 'H04', 'H02-2026-04-01']


   The password recovery email for Atlas stops working after 24 hours.


   The password recovery email for Atlas stops working after 24 hours.


   The password recovery email for Atlas stops working after 24 hours.


The first question gets both versions, H02 and the April document, and the model refuses in 3 of 3: its evidence contradicts itself. The second gets H01, H04 and the April document; today's H02 ranks fourth, outside the three, and all three answers say *The password recovery email for Atlas stops working after 24 hours.* The answer is faithful to its evidence, and wrong under today's policy. The update did what it was asked:
adding a document replaces only a document with the same ID, and the April one has another. Both are in the store:

In [15]:
for document in store.get_by_ids(["H02", "H02-2026-04-01"]):
    print(document.id, document.metadata["version"], "|", document.page_content)

H02 2026-10-01 | Atlas reset links expire after 30 minutes. Request a new link if the old link has expired.
H02-2026-04-01 2026-04-01 | Atlas reset links expire after 24 hours. Request a new link if the old link has expired.


To find every outdated document, compare each version with the one its owner publishes today, in the policy file or
the records file; the chunks have no version:

In [16]:
policies = json.loads((ROOT / "data" / "support-case.json").read_text(encoding="utf-8"))
CURRENT = {doc["source_id"]: doc["version"] for doc in policies + constructed["restricted"]}

def outdated(store):
    """The IDs of documents whose version is not the current version of their source."""
    return [doc["id"] for doc in store.store.values()
            if doc["metadata"].get("version") and doc["metadata"]["version"] != CURRENT[doc["metadata"]["source_id"]]]

print("outdated:", outdated(store))

outdated: ['H02-2026-04-01']


Delete it by its ID, and check that it is gone:

In [17]:
store.delete(["H02-2026-04-01"])
print("found:", [document.id for document in store.get_by_ids(["H02", "H02-2026-04-01"])],
      "| outdated:", outdated(store), "|", len(store.store), "documents")

found: ['H02'] | outdated: [] | 281 documents


Ask the second question again:

In [18]:
q_hits = lc_pipeline.retrieve(store, expiry[1], filter=can_read("customer"))
print([document.id for document, _ in q_hits])
for _ in range(3):
    print("  ", lc_pipeline.answer(expiry[1], lc_pipeline.evidence(q_hits), settings).answer)

['H01', 'H04', 'H02']


   The password recovery email for Atlas stops working after 30 minutes.


   The password recovery email for Atlas stops working after 30 minutes.


   The password recovery email for Atlas stops working after 30 minutes.


Today's H02 is third, and all three answers say *30 minutes*. Deleting by ID works when you know the old ID. When an update keeps the ID, `add_documents` replaces the
old document in place. Add today's H02 again:

In [19]:
h02 = [record for record in records if record["chunk_id"] == "H02"]
store.add_documents(lc_pipeline.to_documents(h02))
print(len(store.store), "documents | H02 version", store.get_by_ids(["H02"])[0].metadata["version"])

281 documents | H02 version 2026-10-01


Still 281 documents: the same ID replaced the document instead of adding one. So give each document an ID that stays
the same across its versions, and when an ID changes, delete the old one in the same update. `outdated()` becomes a
release test in Part E, so a change cannot ship with an old version in its index.

### Checkpoint C

| You should see | Our run |
|---|---|
| April's H02 added | 282 documents |
| Both versions in the store | first question: H02, H02-2026-04-01, H04, *Insufficient evidence.* 3 of 3; second: H01, H04, H02-2026-04-01, *24 hours* 3 of 3 |
| The version check | `outdated: ['H02-2026-04-01']`, then `[]` after `delete`, 281 documents |
| After the delete | H01, H04, H02; *30 minutes* 3 of 3 |
| Same ID added again | 281 documents, H02 version 2026-10-01 |

---

## Part D: Save the rows of a change, with its manifest (self-study)

Part D runs today's release and the change on the same questions and saves what each one did, so tests can read it
later without the store or a key.

The release set is Lab 06's 38 golden records, asked by a customer, plus four cases in the records file: three agent
questions that the runbook or the note answers, and Part B's customer question, which should be refused.

In [20]:
release_set = [{**record, "user": "customer"} for record in golden] + constructed["cases"]
print(len(release_set), "questions |", Counter(record["user"] for record in release_set))
for record in constructed["cases"]:
    print(record["question_id"], record["user"], record["references"] or "refuse", "|", record["question"])

42 questions | Counter({'customer': 39, 'agent': 3})
A01 agent ['4 hours'] | How long can a support agent make an Atlas reset link last?
A02 agent ['admin console'] | How do support agents reset a temporary Atlas account?
A03 agent ['request time'] | What should I check when a customer says an Atlas reset link expired too soon?
A04 customer refuse | Can support send me a new Atlas reset link?


One **row** per question holds what a test needs: who asked, the IDs packed, the answer, whether it passed, the claims
and the evidence lines. It is Lab 06's `evals.score()` plus the user and a filter for that user,
which `score()` does not take:

In [21]:
def row(record, filter_for):
    """One question through the pipeline, searched with filter_for(user); a record with a pack uses those documents."""
    if "pack" in record:
        hits = [(document, None) for document in store.get_by_ids(record["pack"])]
    else:
        hits = lc_pipeline.retrieve(store, record["question"], filter=filter_for(record["user"]))
    lines = lc_pipeline.evidence(hits)
    reply = lc_pipeline.answer(record["question"], lines, settings)
    refused = reply.answer.strip() == evals.REFUSAL
    passed = refused if record["refuse"] else evals.found(reply.answer, record["references"])
    return {"question_id": record["question_id"], "user": record["user"], "type": record["type"],
            "refuse": record["refuse"], "packed": [document.id for document, _ in hits], "answer": reply.answer,
            "refused": refused, "passed": passed, "claims": [claim.model_dump() for claim in reply.claims],
            "evidence": lines}

A row says what happened; the **build manifest** says what produced it. It lists every document in the index with its
source, version, access label and a short hash of its text, then the search settings, a hash of the answer rules, the
release set's version (Lab 06's `version()`) and the stamp. With it, a test can check the index a run used, and anyone can rebuild
that run later:

In [22]:
import hashlib
from shared.answer import RULES

def sha(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:12]

def manifest(store, search):
    documents = [{"id": doc["id"], "source_id": doc["metadata"]["source_id"], "version": doc["metadata"].get("version"),
                  "access": doc["metadata"]["access"], "text": sha(doc["text"])} for doc in store.store.values()]
    return {"search": search, "k": 3, "rules": sha(RULES), "questions": evals.version(release_set, settings)["records"],
            **stamp(settings), "documents": documents}

print({name: value for name, value in manifest(store, "example").items() if name != "documents"})

{'search': 'example', 'k': 3, 'rules': '4be8e10f41b6', 'questions': '3b9ff274df1a', 'chat_model': 'gpt-4o-mini', 'reasoning_effort': None, 'embed_model': 'text-embedding-3-small', 'judge_model': 'gpt-4o-mini', 'rerank_model': 'gpt-4o-mini', 'chat_endpoint': 'https://api.openai.com/v1', 'embed_endpoint': 'https://api.openai.com/v1', 'run_date': '2026-10-24'}


Today's release lets every user read what a customer may read. The change gives agents the runbook and the note; its
filter came from a teammate, and it hides from customers every document labelled `staff`:

In [23]:
def today(user):
    return can_read("customer")

def hide_staff(user):
    """The change's filter: a customer never sees a document labelled staff."""
    return lambda document: user != "customer" or document.metadata["access"] != "staff"

Predict: does `hide_staff` keep every restricted record away from a customer? Run both on the 42 questions (84 calls)
and save each run as a **results file**: the manifest and the rows, in `results/`:

In [24]:
from shared.settings import chat_url

RESULTS = LAB / "results"

def run(name, filter_for, search):
    rows = [row(record, filter_for) for record in release_set]
    results = {"manifest": manifest(store, search), "rows": rows}
    (RESULTS / f"{name}.json").write_text(json.dumps(results, indent=1, ensure_ascii=False), encoding="utf-8")
    print(f"{name:9} passed {sum(r['passed'] for r in rows)} of {len(rows)} | saved results/{name}.json")
    return rows

baseline = run("baseline", today, "every user reads public documents, k=3")
candidate = run("candidate", hide_staff, "customers: no staff documents; agents: all, k=3")
print(f"\n{settings['CHAT_MODEL']} at {chat_url(settings)}")

baseline  passed 35 of 42 | saved results/baseline.json


candidate passed 37 of 42 | saved results/candidate.json

gpt-4o-mini at https://api.openai.com/v1


The change passes 37 of 42, today's release 35. Which questions moved? Lab 08's `paired()`
compares the two runs question by question:

In [25]:
from shared.hybrid import paired

change = paired([r["passed"] for r in baseline], [r["passed"] for r in candidate])
print("both passed", change["both"], "| both failed", change["neither"])
print("gained:", [baseline[i]["question_id"] for i in change["gained"]])
print("lost:  ", [(baseline[i]["question_id"], candidate[i]["answer"]) for i in change["lost"]])

both passed 34 | both failed 4
gained: ['A01', 'A02', 'A03']
lost:   [('G02', 'Insufficient evidence.')]


34 questions pass in both runs and 4 fail in both. The change gains the three agent questions, A01 to A03, which today's release cannot answer, and loses one, G02, now *Insufficient evidence.* On its average, the change is better. Part E asks whether it may ship.

### Checkpoint D

| You should see | Our run |
|---|---|
| The release set | 42 questions: 39 customer, 3 agent |
| Saved | `results/baseline.json`, `results/candidate.json` |
| Passed | today's release 35 of 42, the change 37 |
| Paired | both passed 34, both failed 4; gained A01, A02, A03; lost G02 |

---

## Part E: Write the release tests and run them with pytest (self-study)

Part E turns the boundaries into tests that read the saved files, and runs them on the change.

Start with the check by hand, on the saved results file: is every packed document one the row's user may read? The
labels come from the file's manifest, so the check reads the index the run used, not today's store:

In [26]:
saved = json.loads((RESULTS / "candidate.json").read_text(encoding="utf-8"))
access = {doc["id"]: doc["access"] for doc in saved["manifest"]["documents"]}
for r in saved["rows"]:
    for doc_id in r["packed"]:
        if access[doc_id] not in USERS[r["user"]]:
            print(f"{r['question_id']} ({r['user']}) read {doc_id} ({access[doc_id]}) | {r['answer']}")

G01 (customer) read R04 (internal) | Atlas reset links expire after 30 minutes.
G02 (customer) read R04 (internal) | Insufficient evidence.
G05 (customer) read R04 (internal) | No, temporary Atlas accounts cannot use the usual email route.
G07 (customer) read R04 (internal) | Insufficient evidence.
G09 (customer) read R04 (internal) | Insufficient evidence.
A04 (customer) read R04 (internal) | Insufficient evidence.


Six customer packs hold R04, the incident note: G01, G02, G05, G07, G09 and A04. `hide_staff` names the label `staff`; R04 carries `internal`, a label the teammate never named. An allow-list
such as `can_read` fails the other way: a label it does not list is hidden until someone adds it.

Read the replies: *Atlas reset links expire after 30 minutes.*, *No, temporary Atlas accounts cannot use the usual email route.* and four *Insufficient evidence.* None repeats the note's request time or admin console. A check on the replies would pass this change. The note still went to the model, and
into any trace or log of those requests; only a check on the packs sees it.

**pytest** runs every function whose name starts with `test_` in a file whose name starts with `test_`, and reports
each `assert` that fails with its message ([pytest docs](https://docs.pytest.org/en/stable/getting-started.html)).
The release tests are in `test_release.py` in this folder. Read the first one:

In [27]:
import inspect

sys.path.insert(0, str(LAB))
import test_release

print(inspect.getsource(test_release.test_no_restricted_source_in_pack))

def test_no_restricted_source_in_pack(candidate):
    access = {doc["id"]: doc["access"] for doc in candidate["manifest"]["documents"]}
    leaks = [f"{row['question_id']} ({row['user']}) read {doc_id} ({access[doc_id]})"
             for row in candidate["rows"] for doc_id in row["packed"] if access[doc_id] not in USERS[row["user"]]]
    assert not leaks, f"{len(leaks)} packed documents the user may not read: " + "; ".join(leaks)



It is the check above, with the rows and the manifest read from a results file. The argument `candidate` is a
**fixture**: pytest calls the function of that name in the file, which loads the file named by `GATE_CANDIDATE`
(`results/candidate.json` when it is not set), and passes in what it returns. `USERS` is read from the records file, so
the gate holds the permission rules, not the change.

Run the tests as a CI job would: in a new Python process, on the saved file, with the key set to empty. `gate()` starts
`pytest` from the repository folder; `-k restricted` runs only the tests whose names contain *restricted*, and
`--tb=short` keeps each failure to its assertion:

In [28]:
import os
import subprocess

def gate(candidate_file, *options):
    """Run the release tests on a saved results file, in a new process, as a CI job would."""
    command = [sys.executable, "-m", "pytest", "11_Release_Gate/test_release.py", "-q", "--tb=short", "--color=no", *options]
    env = {**os.environ, "GATE_CANDIDATE": str(candidate_file), "OPENAI_API_KEY": ""}   # no key needed
    print(subprocess.run(command, cwd=ROOT, env=env, capture_output=True, text=True).stdout)

gate(RESULTS / "candidate.json", "-k", "restricted")

F                                                                        [100%]
=================================== FAILURES ===================================
______________________ test_no_restricted_source_in_pack _______________________
11_Release_Gate/test_release.py:50: in test_no_restricted_source_in_pack
    assert not leaks, f"{len(leaks)} packed documents the user may not read: " + "; ".join(leaks)
E   AssertionError: 6 packed documents the user may not read: G01 (customer) read R04 (internal); G02 (customer) read R04 (internal); G05 (customer) read R04 (internal); G07 (customer) read R04 (internal); G09 (customer) read R04 (internal); A04 (customer) read R04 (internal)
E   assert not ['G01 (customer) read R04 (internal)', 'G02 (customer) read R04 (internal)', 'G05 (customer) read R04 (internal)', 'G07 (customer) read R04 (internal)', 'G09 (customer) read R04 (internal)', 'A04 (customer) read R04 (internal)']
=========================== short test summary info ==============

`1 failed, 4 deselected`: the access test failed, and the line after `AssertionError` names each pack that broke it.
The other four tests:

In [29]:
for test in (test_release.test_only_current_versions_in_index, test_release.test_refusal_slice_held,
             test_release.test_regressions_reviewed, test_release.test_quotes_found):
    print(inspect.getsource(test))

def test_only_current_versions_in_index(candidate):
    stale = [f"{doc['id']} ({doc['source_id']} {doc['version']}, current {CURRENT[doc['source_id']]})"
             for doc in candidate["manifest"]["documents"]
             if doc["version"] and doc["version"] != CURRENT[doc["source_id"]]]
    assert not stale, "outdated versions in the index: " + "; ".join(stale)

def test_refusal_slice_held(baseline, candidate):
    held = {row["question_id"] for row in baseline["rows"] if row["refuse"] and row["passed"]}
    broken = [f"{row['question_id']}: {row['answer']}" for row in candidate["rows"]
              if row["question_id"] in held and not row["passed"]]
    assert not broken, "refusals the baseline held, now answered: " + "; ".join(broken)

def test_regressions_reviewed(baseline, candidate):
    passed_before = {row["question_id"] for row in baseline["rows"] if row["passed"]}
    lost = [f"{row['question_id']}: {row['answer']}" for row in candidate["rows"]
            if row["ques

| Test | Fails when | Kind |
|---|---|---|
| `test_no_restricted_source_in_pack` | a pack holds a document its user may not read | hard: never allowed |
| `test_only_current_versions_in_index` | the index holds a document older than its source's current version (Part C) | hard |
| `test_refusal_slice_held` | the change answers a question the baseline rightly refused | hard |
| `test_regressions_reviewed` | a question the baseline passed fails now, and nobody reviewed it | reviewed: a person may accept it |
| `test_quotes_found` | under `MIN_QUOTES_FOUND` (95%) of the quotes are in the lines they cite (Lab 02) | threshold |

A reviewed regression goes in `REVIEWED`, the dictionary at the top of the file, with the reason, in a reviewed
commit: a loss can be accepted, but only by someone who read the row. Run all five:

In [30]:
gate(RESULTS / "candidate.json")

F..F.                                                                    [100%]
=================================== FAILURES ===================================
______________________ test_no_restricted_source_in_pack _______________________
11_Release_Gate/test_release.py:50: in test_no_restricted_source_in_pack
    assert not leaks, f"{len(leaks)} packed documents the user may not read: " + "; ".join(leaks)
E   AssertionError: 6 packed documents the user may not read: G01 (customer) read R04 (internal); G02 (customer) read R04 (internal); G05 (customer) read R04 (internal); G07 (customer) read R04 (internal); G09 (customer) read R04 (internal); A04 (customer) read R04 (internal)
E   assert not ['G01 (customer) read R04 (internal)', 'G02 (customer) read R04 (internal)', 'G05 (customer) read R04 (internal)', 'G07 (customer) read R04 (internal)', 'G09 (customer) read R04 (internal)', 'A04 (customer) read R04 (internal)']
__________________________ test_regressions_reviewed _____________

`2 failed, 3 passed`. The second failure is G02, which today's release answered: in the change's pack R04 took second place, H02 dropped out of the three, and the model refused. Do not add G02 to `REVIEWED`: the cause is the leak, so the fix belongs in the change, not in the
gate. No test reads the average: a higher pass rate cannot buy back a document a customer should never have read.

### Checkpoint E

| You should see | Our run |
|---|---|
| By hand | R04 in six customer packs: G01, G02, G05, G07, G09, A04 |
| `-k restricted` | `1 failed, 4 deselected`; the assertion lists the six packs |
| All five | `2 failed, 3 passed`: the access test (six packs) and the regressions (G02, *Insufficient evidence.*) |

---

## Part F: Fix the change and pass the gate (self-study)

Part F fixes the cause and runs the gate again.

The fix replaces the change's filter with `can_read`, the allow-list from Part B. Run the release set and the gate:

In [31]:
fixed = run("fixed", can_read, "allow-list per user, k=3")
gate(RESULTS / "fixed.json")

fixed     passed 38 of 42 | saved results/fixed.json


.....                                                                    [100%]
5 passed in 0.02s



The fixed change passes 38 of 42, and the gate says `5 passed`: no pack holds a document its user may not read, and G02 passes again. Compare the fixed change with today's release:

In [32]:
change = paired([r["passed"] for r in baseline], [r["passed"] for r in fixed])
print("gained:", [baseline[i]["question_id"] for i in change["gained"]])
print("lost:  ", [(baseline[i]["question_id"], fixed[i]["answer"]) for i in change["lost"]])

gained: ['A01', 'A02', 'A03']
lost:   []


Gained A01, A02 and A03, lost nothing. A passing gate says that nothing the release forbids happened on these questions. Whether the change is
worth shipping is the paired comparison's call (Lab 08): here the agents gain and the customers lose nothing.

The gate says no more than its five tests read. `test_quotes_found` checks that a quote's words are in the line it
cites, not that the line means what the claim says (Lab 07's judge passed answers that quoted real lines). Its 95% is
a threshold we chose: the fixed change has 40 claims, so two could fail and the gate would still pass. And the tests
read the 42 saved questions: a leak or a lost answer on a question outside the release set is caught only once that
question is added to it.

The same file runs anywhere the results files are, with no key, which is what a CI job needs. From the repository
folder:

```bash
GATE_BASELINE=path/to/baseline.json GATE_CANDIDATE=path/to/candidate.json python -m pytest 11_Release_Gate/test_release.py -q
```

The capstone runs it on its own service's results.

### Checkpoint F

| You should see | Our run |
|---|---|
| The fixed change | passed 38 of 42; `results/fixed.json` saved |
| The gate | `5 passed` |
| Paired with today's release | gained A01, A02, A03; lost none |

---

## What you can do now

- Label every document with who may read it, and list a user's allowed records with an allow-list.
- Filter by permission after the search and before it, and say why a post-filter can leave a customer no evidence.
- Replace an outdated policy by ID with `delete`, `add_documents` and `get_by_ids`, and check that no old version is
  left in the index.
- Write a build manifest and save one row per question for a release and for a change.
- Write release tests with pytest, run them on saved results with no key, and read a failing assertion back to its
  rows.
- Explain why a change with a higher average still fails on one leaked document, and when a regression may be
  reviewed instead of fixed.

## Going further: let the index remove old versions

Part C found the April document with a version check and deleted it by hand. LangChain's indexing API does this as
part of every update ([LangChain: indexing](https://reference.langchain.com/python/langchain-core/indexing/api/index)): a
**record manager** remembers which documents came from which source, and with `cleanup="incremental"` each new
version of a source deletes the old one. Index the five policies as they were in April, then as they are today:

In [33]:
from langchain_core.indexing import InMemoryRecordManager, index
from langchain_core.vectorstores import InMemoryVectorStore
from shared.settings import lc_embeddings

policy_store = InMemoryVectorStore(lc_embeddings(settings))
manager = InMemoryRecordManager("atlas-policies")
manager.create_schema()
october = lc_pipeline.to_documents([{**policy, "chunk_id": policy["source_id"]} for policy in policies])
april_docs = lc_pipeline.to_documents([april]) + [doc for doc in october if doc.id != "H02"]
for name, documents in [("April", april_docs), ("October", october)]:
    print(name, index(documents, manager, policy_store, cleanup="incremental", source_id_key="source_id",
                      key_encoder="sha256"))
print(sorted((doc["metadata"]["source_id"], doc["metadata"]["version"]) for doc in policy_store.store.values()))

April {'num_added': 5, 'num_updated': 0, 'num_skipped': 0, 'num_deleted': 0}


October {'num_added': 1, 'num_updated': 0, 'num_skipped': 4, 'num_deleted': 1}
[('H01', '2026-10-01'), ('H02', '2026-10-01'), ('H03', '2026-10-01'), ('H04', '2026-10-01'), ('H05', '2026-10-01')]


The April run added five documents. The October run added the new H02, skipped the four policies that had not changed and deleted the April H02 (`num_deleted: 1`), so each source keeps its current version. The record manager keeps its own IDs (hashes), so the store's IDs are no longer `H01` to `H05`.

## Challenge

Use only what this lab showed. A worked solution is in `solved/` for the instructor. Task 1 is the challenge; task 2 is optional. For each task, keep the printed results and your answer to
**Explain**.

1. **Ship the fixed change with April's H02 left in.** Part C deleted the April document by hand. What does the gate
   say when nobody does?

   - Under **Task 1**, add the April document back, run the release set with `can_read` as `"april"`, run `gate()` on
     its file, and delete the April document again.

   **Check:** pytest names the version test and the April document's ID. Our worked solution passed 36 of 42 and got `2 failed, 3 passed`: the version test named `H02-2026-04-01 (H02 2026-04-01, current 2026-10-01)`, and the regression test named G01 (*Insufficient evidence.*) and G02 (*... stops working after 24 hours.*).

   **Explain:** which of the failing tests would still fail if the April document never reached a pack?

2. **Optional: check the citations instead of the packs.** Would a test that reads the cited IDs have caught the
   change in Part D?

   - Complete `cited_unreadable()` under **Task 2**: the `(question_id, source_id)` of every claim whose source the
     row's user may not read. Run it on `candidate` and on `fixed`.

   **Check:** two lists. Our worked solution found none in either: `candidate: []` and `fixed: []`.

   **Explain:** what does a citation check read, and what did the model read?

**Task 1.** Complete the cell, then run it:

In [34]:
# TODO: add the April document back with store.add_documents(lc_pipeline.to_documents([april])),
#       run the release set: run("april", can_read, "allow-list per user, k=3, April H02 left in"),
#       run the gate on it: gate(RESULTS / "april.json"),
#       then delete the April document again with store.delete(["H02-2026-04-01"])
print("outdated after the delete:", outdated(store))

outdated after the delete: []


**Task 2 (optional).** Complete `cited_unreadable()`, then run the cell:

In [35]:
def cited_unreadable(rows):
    """(question_id, source_id) for every claim whose source the row's user may not read."""
    # TODO: loop over the rows and their claims; keep a claim when access.get(claim["source_id"], "unknown")
    #       is not in USERS[row["user"]]
    return None

print("candidate:", cited_unreadable(candidate))
print("fixed:    ", cited_unreadable(fixed))

candidate: None
fixed:     None


## Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| `KeyError: 'access'` in `can_read` | the store was built from `load_corpus()` without Part A's labels | Run Part A's `records` cell and build the store from `records` |
| `FileNotFoundError: [Errno 2] No such file or directory: '.../results/candidate.json'` in pytest's output | the results file was not saved yet, or `GATE_CANDIDATE` names a path that does not exist | Run Part D first, or check the path you set |
| `ERROR: file or directory not found: 11_Release_Gate/test_release.py` | pytest was started from inside the lab folder with the path from the repository folder | Start it from the repository folder, or run `pytest test_release.py` inside the lab folder |

## Resources

- [LangChain: vector stores](https://docs.langchain.com/oss/python/integrations/vectorstores): `add_documents` with
  IDs, `delete`, `get_by_ids` and metadata filters
- [LangChain: indexing API](https://reference.langchain.com/python/langchain-core/indexing/api/index): the record manager and
  `cleanup` modes that remove old versions
- [pytest: get started](https://docs.pytest.org/en/stable/getting-started.html) and
  [fixtures](https://docs.pytest.org/en/stable/how-to/fixtures.html): test discovery, `assert` messages and fixtures
- [OWASP: LLM02 sensitive information disclosure](https://genai.owasp.org/llmrisk/llm022025-sensitive-information-disclosure/):
  why access control belongs before the model, not in the prompt

Next: [the capstone](../Capstone_RAG_Service/README.md), *How to ship a RAG service with its release gate*. Back to [the lab list](../README.md).